# Clip planes

Port of `packages/niivue/examples/vox.clip.html`. Loads the MNI template and exposes clip-plane count, cutaway mode, and clip-plane color controls.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

def planes_for(count):
    if count == 0:
        return [[2.0, 180, 20]]
    if count == 1:
        return [[0.1, 180, 20]]
    if count == 2:
        return [[0.1, 180, 20], [0.1, 0, -20]]
    if count == 3:
        return [[0.0, 90, 0], [0.0, 0, -20], [0.1, 0, -90]]
    if count == 4:
        return [[0.3, 270, 0], [0.3, 90, 0], [0.0, 180, 0], [0.1, 0, 0]]
    if count == 5:
        return [[0.4, 270, 0], [0.4, 90, 0], [0.4, 180, 0], [0.2, 0, 0], [0.1, 0, -90]]
    return [[0.4, 270, 0], [-0.1, 90, 0], [0.4, 180, 0], [0.2, 0, 0], [0.1, 0, -90], [0.3, 0, 90]]

nv = NiiVue(slice_type=4, backend="webgl2")

clip_count = widgets.Dropdown(options=list(range(7)), value=2, description="Planes")
cutaway = widgets.Checkbox(value=False, description="Cutaway")
clip_color = widgets.Dropdown(
    options=[("Transparent", 0.0), ("Translucent", 0.3), ("Shading", -0.2)],
    value=0.3,
    description="Color",
)

def update_planes(change=None):
    nv.set_clip_planes(planes_for(clip_count.value))

def update_cutaway(change):
    nv.is_clip_plane_cutaway = change["new"]

def update_clip_color(change):
    color = nv.clip_plane_color
    if not isinstance(color, (list, tuple)) or len(color) != 4:
        color = [0.4, 0.1, 0.7, 0.3]
    nv.clip_plane_color = [color[0], color[1], color[2], change["new"]]

clip_count.observe(update_planes, names="value")
cutaway.observe(update_cutaway, names="value")
clip_color.observe(update_clip_color, names="value")

controls = widgets.HBox([clip_count, cutaway, clip_color])
display(widgets.VBox([controls, nv]))

nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
update_clip_color({"new": clip_color.value})
update_planes()
